### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

Tracking agent behavior with logging, analytics, and debugging.
Transforming prompts, tool selection, and output formatting.
Adding retries, fallbacks, and early termination logic.
Applying rate limits, guardrails, and PII detection.

In [21]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")
from IPython.display import display, Markdown

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

Long-running conversations that exceed context windows.
Multi-turn dialogues with extensive history.
Applications where preserving full conversation context matters.

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

When you pass <b>config={"configurable": {"thread_id": "test-1"}}</b> into an agent, you are explicitly activating <b>Multi-User</b> Chat Isolation (Sessions).

The config object functions like a database query filter, telling LangGraph: "Go pull up the exact conversation timeline belonging to User ID X, load their historical context, append this new message to it, and process the response."

In [23]:
config1 = {"configurable":{"thread_id":"user1"}}
config2 = {"configurable":{"thread_id":"user2"}}


<b>checkpointer=InMemorySaver():</b> This activates the graph's memory tracking. It tells LangGraph to save the conversation state into RAM so it can remember things across different chat turns on a specific thread_id.

In [ ]:
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: [HumanMessage(content='remember code 97', additional_kwargs={}, response_metadata={}, id='1a980b7b-f12f-46ca-9b96-14b1182044d8'), AIMessage(content=[{'type': 'text', 'text': 'Understood. I have noted **code 97**. I will keep this in mind for our future interactions. \n\nLet me know whenever you need me to reference it!', 'extras': {'signature': 'EjQKMgEMOdbHt5TFTQmDAMxoIRJsRCWAJACOTXtZ2jVykZ6rEUh4Hd+qkpOUlcmXfMyOf42j'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e5f31-6b4a-7e41-993a-90d55b66a368-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 36, 'total_tokens': 42, 'input_token_details': {'cache_read': 0}})]
Messages: 2
Messages: [HumanMessage(content='what is the code i told you to remember?', additional_kwargs={}, response_metadata={}, id='7022a3ab-16d6-4234-aab8-46f12d9f2220'), AIMessage(conte

### I represented how two configs don't remeber each other's conversation

In [25]:
response=agent.invoke({"messages":["remember code 97"]},config1)
print(f"Messages: {response["messages"]}")
print(f"Messages: {len(response['messages'])}")


response = agent.invoke({"messages":["what is the code i told you to remember?"]}, config2)
print(f"Messages: {response["messages"]}")
print(f"Messages: {len(response['messages'])}")

Messages: [HumanMessage(content='remember code 97', additional_kwargs={}, response_metadata={}, id='1a980b7b-f12f-46ca-9b96-14b1182044d8'), AIMessage(content=[{'type': 'text', 'text': 'Understood. I have noted **code 97**. I will keep this in mind for our future interactions. \n\nLet me know whenever you need me to reference it!', 'extras': {'signature': 'EjQKMgEMOdbHt5TFTQmDAMxoIRJsRCWAJACOTXtZ2jVykZ6rEUh4Hd+qkpOUlcmXfMyOf42j'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e5f31-6b4a-7e41-993a-90d55b66a368-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 36, 'total_tokens': 42, 'input_token_details': {'cache_read': 0}}), HumanMessage(content='remember code 97', additional_kwargs={}, response_metadata={}, id='a02b7fb7-e56f-48d8-8371-f494b420e25a'), AIMessage(content=[{'type': 'text', 'text': 'Acknowledged. I h

### Token Size

In [38]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city:str):
    """ Search hotels - returns long response to use more tokens """
    return f""" 
        Hotels in {city}:
        1. Grand Hotel - 5 star, $350/night, spa, pool, gym
        2. City Inn - 4 star, $180/night, business center
        3. Budget Stay - 3 star, $75/night, free wifi"
    """

config = {"configurable":{"thread_id":"user1"}}

agent = create_agent(
    model = "google_genai:gemini-3.1-flash-lite",
    checkpointer = InMemorySaver(),
    middleware=
        [SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("tokens",500),        
            keep=("tokens",300)
        )],
    tools=[search_hotels])

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4

for city in cities:
    response = agent.invoke({"messages":[f"find the hotels in the {city}"]}, config=config)
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~151 tokens, 4 messages
[HumanMessage(content='find the hotels in the Paris', additional_kwargs={}, response_metadata={}, id='409c2ab6-529c-4298-ac5a-02d317af8893'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'2ab869aa-4016-489b-9e13-5d2d9776e4ca': 'EjQKMgEMOdbHppnwKNoelCO0172TIx5EMoxJ07FyNPjBQBPFk2Ko9JAU7EkxtAE1+Wip9qPv'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e5f66-ab3f-7101-b338-86e7f0d55f3e-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': '2ab869aa-4016-489b-9e13-5d2d9776e4ca', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 16, 'total_tokens': 70, 'input_token_details': {'cache_read': 0}}), ToolMessage(content=' \n        Hotels in Paris:\n        1. Grand Hote

### Why the Summary is given as HumanMessage in subsequesnt requests??
Many modern chat models (especially when hosted behind cloud APIs) treat the system prompt as a rigid, unchangeable instruction set given at the start of a thread. If a middleware script constantly injects new, changing SystemMessages into the chat history, some LLM providers either ignore them completely or get confused because they expect only one structural system prompt at the very beginning.

### FRACTION

In [40]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])


Paris: ~89 tokens (0.0695%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='a32fce48-f02f-42f5-8474-83229c001741'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_hotels', 'arguments': '{"city": "Paris"}'}, '__gemini_function_call_thought_signatures__': {'dceec7bb-e563-4ac9-a77f-e03470d31dca': 'EjQKMgEMOdbHAOB52ICSxuIEzCnUd2pU+zq0Q9MXhNtDop6EfqgWUUY6hKDQaR6TviM39snD'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e5f7d-ea19-7542-9aa3-f342f53e419c-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'dceec7bb-e563-4ac9-a77f-e03470d31dca', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 16, 'total_tokens': 61, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='Hotels in Paris: Grand Hotel $350, City Inn $180, Budget S

### Human-in-the-loop

In [42]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 
from langchain.tools import tool


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[read_email_tool, send_email_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email_tool": False,  
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"]
                    },  
            },           
        ),
    ],
    checkpointer=InMemorySaver(),
)

In [43]:
config = {"configurable": {"thread_id": "test-approve"}}
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='ff09a330-1c2d-450e-97d6-ee43e57d7139'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'send_email_tool', 'arguments': '{"recipient": "john@test.com", "subject": "Hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'741ba0e3-f0b1-4279-9dcf-3c732429224c': 'EjQKMgEMOdbHxD8tl1Prk9Ye9Ots458ayQNTfzBoVgHUnpfzLtBzby1c/vUYVJbA27y8Vfki'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e5fa1-3a8c-7510-8805-22e54afb5575-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': '741ba0e3-f0b1-4279-9dcf-3c732429224c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 144, 'output_t

In [ ]:
# agent.invoke(
#     Command(resume={
#         "decisions": [
#             {"type": "approve"},  # Decision for Tool Call 1
#             {"type": "approve"},  # Decision for Tool Call 2
#             {"type": "reject"}   # Decision for Tool Call 3
#         ]
#     }), 
#     config=config, 
#     version="v2"
# )

In [45]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")


⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "OK. I've sent the email to john@test.com.", 'extras': {'signature': 'EjQKMgEMOdbHfR163ROMgeKTpo5a1y2urFGNigGQVxZ7xC0WnbzzVXmCosr/wn/2yZ6NNSdc'}}]
